# Module 4.1: Gateway and Lambda Tools

Deploy your customer service tools as AWS Lambda functions and expose them through AgentCore Gateway as managed MCP endpoints. This gives your agent secure, scalable tool access without managing infrastructure.

**Prerequisites:** Module 3 completed, AWS credentials configured, CDK installed (`npm install -g aws-cdk`)

---

## Architecture Overview

In Workshop 1, tools ran in-process with the agent. Now we deploy them as independent Lambda functions behind AgentCore Gateway:

```
Agent → MCP Client → AgentCore Gateway → Lambda (lookup_customer)
                                        → Lambda (get_order_history)
                                        → Lambda (process_refund)
```

Gateway handles IAM auth, rate limiting, and MCP protocol translation. Each Lambda is a standalone function with the same logic as our `@tool` functions.

---

## Step 1: Install Dependencies

In [ ]:
!pip install -q --disable-pip-version-check boto3 strands-agents bedrock-agentcore

In [ ]:
# Where this module fits in the harness you are building
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
from workshop.workshop_utils import lego_progress, quiet_logs

quiet_logs()          # keep third-party SDK logging out of the teaching output
lego_progress(3)       # the harness tower so far - one brick per module

---

## Step 2: Review the Lambda Handlers

Each tool is a simple Lambda handler that takes an event dict and returns a result. Same logic as Workshop 1, just packaged for Lambda.

In [ ]:
# Let's look at the lookup_customer handler
print(open("lambda_tools/lookup_customer/handler.py").read())

In [ ]:
# The get_order_history handler
print(open("lambda_tools/get_order_history/handler.py").read())

In [ ]:
# The process_refund handler
print(open("lambda_tools/process_refund/handler.py").read())

---

## Step 3: Deploy the Lambda Tools with boto3 (primary path)

This is the recommended path inside the Code Editor — it needs **no CDK bootstrap and no terminal**. The cell below:

1. Creates a least-privilege Lambda **execution role** (`customer-service-lambda-role`) trusted by `lambda.amazonaws.com` with the AWS-managed `AWSLambdaBasicExecutionRole` (CloudWatch Logs only).
2. Packages each `lambda_tools/<tool>/handler.py` into an in-memory ZIP (the handlers use only the Python standard library, so no dependencies to bundle).
3. Creates each function, or updates its code if it already exists — so the cell is **idempotent** and safe to re-run.

> A brief retry loop handles IAM role propagation: a freshly created role can take a few seconds before Lambda is allowed to assume it.

> Prefer infrastructure-as-code? The same three functions are defined in `cdk/stack.py` — see the **CDK alternative** later in this notebook.

In [ ]:
import io
import json
import time
import zipfile

import boto3
from botocore.exceptions import ClientError

REGION = "us-east-1"
ROLE_NAME = "customer-service-lambda-role"

# Each tool maps to its handler directory under lambda_tools/. The function
# names match the customer-service-* pattern the workshop IAM policy allows.
TOOLS = {
    "customer-service-lookup-customer": {
        "dir": "lambda_tools/lookup_customer",
        "description": "Looks up a customer by ID",
    },
    "customer-service-get-order-history": {
        "dir": "lambda_tools/get_order_history",
        "description": "Gets order history for a customer",
    },
    "customer-service-process-refund": {
        "dir": "lambda_tools/process_refund",
        "description": "Processes a refund for an order",
    },
}

iam = boto3.client("iam", region_name=REGION)
lambda_client = boto3.client("lambda", region_name=REGION)
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]


def ensure_execution_role():
    """Create (or reuse) the Lambda execution role and return its ARN."""
    trust_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Principal": {"Service": "lambda.amazonaws.com"},
                "Action": "sts:AssumeRole",
            }
        ],
    }
    try:
        role = iam.create_role(
            RoleName=ROLE_NAME,
            AssumeRolePolicyDocument=json.dumps(trust_policy),
            Description="Execution role for customer service Lambda tools",
        )
        print(f"Created role {ROLE_NAME}")
    except iam.exceptions.EntityAlreadyExistsException:
        role = iam.get_role(RoleName=ROLE_NAME)
        print(f"Reusing existing role {ROLE_NAME}")

    # AWSLambdaBasicExecutionRole grants only CloudWatch Logs write access.
    iam.attach_role_policy(
        RoleName=ROLE_NAME,
        PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole",
    )
    return role["Role"]["Arn"]


def zip_handler(handler_dir):
    """Package a handler directory into an in-memory ZIP for Lambda."""
    buffer = io.BytesIO()
    with zipfile.ZipFile(buffer, "w", zipfile.ZIP_DEFLATED) as zf:
        with open(f"{handler_dir}/handler.py") as f:
            zf.writestr("handler.py", f.read())
    buffer.seek(0)
    return buffer.read()


def deploy_function(name, config, role_arn):
    """Create the function, or update its code if it already exists."""
    code_zip = zip_handler(config["dir"])

    # IAM role propagation can lag function creation by a few seconds.
    for attempt in range(6):
        try:
            lambda_client.create_function(
                FunctionName=name,
                Runtime="python3.12",
                Role=role_arn,
                Handler="handler.handler",
                Code={"ZipFile": code_zip},
                Timeout=10,
                MemorySize=128,
                Description=config["description"],
            )
            print(f"  Created {name}")
            return
        except lambda_client.exceptions.ResourceConflictException:
            lambda_client.update_function_code(FunctionName=name, ZipFile=code_zip)
            print(f"  Updated {name} (already existed)")
            return
        except ClientError as e:
            if e.response["Error"]["Code"] == "InvalidParameterValueException" and attempt < 5:
                time.sleep(5)  # role not assumable yet — wait and retry
                continue
            raise


role_arn = ensure_execution_role()
print(f"Execution role: {role_arn}\n")

print("Deploying functions:")
for fn_name, fn_config in TOOLS.items():
    deploy_function(fn_name, fn_config, role_arn)

print("\n✅ All Lambda tools deployed.")


---

## Step 3 (alternative): Deploy with CDK

Only needed if you chose the CDK path. Run these commands in your terminal from the `cdk/` directory. The instance role already allows the CloudFormation, CDK bootstrap, and `customer-service-*` Lambda/role actions the CDK needs.

In [ ]:
# Deploy the stack (run in terminal from the cdk/ directory)
print("""
Run these commands in your terminal:

  cd cdk/
  pip install -r requirements.txt
  cdk bootstrap   # (first time only)
  cdk deploy

CDK will output the Lambda ARNs after deployment.
""")

---

## Step 4: Register Tools with AgentCore Gateway

Now create a Gateway and register each Lambda as a target. The Gateway exposes them as MCP-compatible endpoints that any agent can discover and call over IAM (SigV4) auth.

The next three cells:
1. Collect the Lambda ARNs.
2. Create a Gateway execution role and the Gateway (`bedrock-agentcore-control`), waiting until it's `READY`.
3. Register each Lambda as a Gateway target with its MCP tool schema.

## CLI alternative

Prefer the terminal? The boto3 cells below are the primary path. With the
`agentcore` starter toolkit (`pip install bedrock-agentcore-starter-toolkit`)
you can create the Gateway from a terminal:

```bash
# Create the MCP Gateway (creates an IAM role if you omit --role-arn)
agentcore gateway create-mcp-gateway \
  --region us-east-1 \
  --name customer-service-gateway

# List / inspect / delete gateways
agentcore gateway list-mcp-gateways --region us-east-1
agentcore gateway get-mcp-gateway   --region us-east-1
```

Registering each Lambda as a target with its tool schema is done with boto3
in the cells below (`create_gateway_target`), which gives full control over the
per-tool `inputSchema`. The toolkit's `agentcore gateway create-mcp-gateway-target`
covers the gateway-arn/url/role-arn wiring; the boto3 path is used here so each
tool's schema is explicit.


In [ ]:
import boto3
import json

# Lambda ARNs for the functions deployed earlier (boto3 or CDK — same names).
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]
REGION = "us-east-1"

LOOKUP_CUSTOMER_ARN = f"arn:aws:lambda:{REGION}:{ACCOUNT_ID}:function:customer-service-lookup-customer"
GET_ORDER_HISTORY_ARN = f"arn:aws:lambda:{REGION}:{ACCOUNT_ID}:function:customer-service-get-order-history"
PROCESS_REFUND_ARN = f"arn:aws:lambda:{REGION}:{ACCOUNT_ID}:function:customer-service-process-refund"

print(f"Account: {ACCOUNT_ID}")
print(f"Region: {REGION}")
print("\nLambda ARNs:")
print(f"  lookup_customer:   {LOOKUP_CUSTOMER_ARN}")
print(f"  get_order_history: {GET_ORDER_HISTORY_ARN}")
print(f"  process_refund:    {PROCESS_REFUND_ARN}")


In [ ]:
# Create the Gateway with boto3.
# Gateway management is a control-plane operation, so it lives on the
# bedrock-agentcore-control client. A Gateway needs an execution role it can
# assume to invoke your Lambdas, plus an authorizer type (AWS_IAM = SigV4).
import time

control_client = boto3.client("bedrock-agentcore-control", region_name=REGION)
iam = boto3.client("iam", region_name=REGION)

GATEWAY_NAME = "customer-service-gateway"
GATEWAY_ROLE_NAME = "customer-service-gateway-role"


def ensure_gateway_role():
    """Create (or reuse) the role the Gateway assumes to invoke the Lambdas."""
    trust = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
                "Action": "sts:AssumeRole",
            }
        ],
    }
    try:
        role = iam.create_role(
            RoleName=GATEWAY_ROLE_NAME,
            AssumeRolePolicyDocument=json.dumps(trust),
            Description="Execution role for the customer service Gateway",
        )
        print(f"Created role {GATEWAY_ROLE_NAME}")
        time.sleep(10)  # let the role propagate before the Gateway assumes it
    except iam.exceptions.EntityAlreadyExistsException:
        role = iam.get_role(RoleName=GATEWAY_ROLE_NAME)
        print(f"Reusing role {GATEWAY_ROLE_NAME}")
    iam.put_role_policy(
        RoleName=GATEWAY_ROLE_NAME,
        PolicyName="invoke-lambdas",
        PolicyDocument=json.dumps({
            "Version": "2012-10-17",
            "Statement": [{
                "Effect": "Allow",
                "Action": "lambda:InvokeFunction",
                "Resource": f"arn:aws:lambda:{REGION}:{ACCOUNT_ID}:function:customer-service-*",
            }],
        }),
    )
    return role["Role"]["Arn"]


def find_gateway_id(name):
    """Return the gatewayId for a gateway by name, paging through all results."""
    next_token = None
    while True:
        kwargs = {"nextToken": next_token} if next_token else {}
        page = control_client.list_gateways(**kwargs)
        for it in page.get("items", []):
            if it["name"] == name:
                return it["gatewayId"]
        next_token = page.get("nextToken")
        if not next_token:
            return None


gateway_role_arn = ensure_gateway_role()

try:
    gw = control_client.create_gateway(
        name=GATEWAY_NAME,
        description="Customer service tools gateway",
        roleArn=gateway_role_arn,
        protocolType="MCP",
        authorizerType="AWS_IAM",
    )
    GATEWAY_ID = gw["gatewayId"]
    GATEWAY_URL = gw["gatewayUrl"]
    GATEWAY_ARN = gw["gatewayArn"]
    print("✅ Gateway created!")
except control_client.exceptions.ConflictException:
    # Already exists — resolve it by name (paginated), then fetch its details.
    GATEWAY_ID = find_gateway_id(GATEWAY_NAME)
    if not GATEWAY_ID:
        raise RuntimeError(
            f"Gateway '{GATEWAY_NAME}' reported as existing but was not found via "
            "list_gateways. Check the AgentCore console or delete the stale gateway."
        )
    details = control_client.get_gateway(gatewayIdentifier=GATEWAY_ID)
    GATEWAY_URL = details["gatewayUrl"]
    GATEWAY_ARN = details["gatewayArn"]
    print("Gateway already exists — reusing it.")

print(f"   ID:  {GATEWAY_ID}")
print(f"   URL: {GATEWAY_URL}")
print(f"   ARN: {GATEWAY_ARN}")

# Targets can only be added once the Gateway leaves CREATING, so wait for READY.
print("\nWaiting for the Gateway to be READY...")
gateway_ready = False
for _ in range(24):
    status = control_client.get_gateway(gatewayIdentifier=GATEWAY_ID)["status"]
    if status == "READY":
        gateway_ready = True
        print("  Gateway READY ✅")
        break
    if status in ("FAILED", "UPDATE_UNSUCCESSFUL"):
        raise RuntimeError(f"Gateway entered {status}")
    time.sleep(5)
if not gateway_ready:
    raise TimeoutError("Gateway did not become READY within ~2 minutes — check the console.")

# Save these for later modules:
print(f"\n👉 Module 4.2 needs the URL:  export AGENTCORE_GATEWAY_URL=\"{GATEWAY_URL}\"")
print(f"👉 Module 5 needs the ARN:  export AGENTCORE_GATEWAY_ARN=\"{GATEWAY_ARN}\"")


In [ ]:
# Register each Lambda as a Gateway target.
# Each target maps one Lambda to its MCP tool schema. The schema shape is the
# Gateway's own format: targetConfiguration.mcp.lambda with an inlinePayload
# tool list, plus a GATEWAY_IAM_ROLE credential provider (the Gateway uses the
# role from the previous cell to invoke the Lambda).
targets = [
    {
        "name": "lookup-customer",
        "lambda_arn": LOOKUP_CUSTOMER_ARN,
        "tool": {
            "name": "lookup_customer",
            "description": "Look up a customer by their ID",
            "inputSchema": {
                "type": "object",
                "properties": {
                    "customer_id": {"type": "string", "description": "The customer ID (e.g. C-1001)"}
                },
                "required": ["customer_id"],
            },
        },
    },
    {
        "name": "get-order-history",
        "lambda_arn": GET_ORDER_HISTORY_ARN,
        "tool": {
            "name": "get_order_history",
            "description": "Get order history for a customer",
            "inputSchema": {
                "type": "object",
                "properties": {
                    "customer_id": {"type": "string", "description": "The customer ID (e.g. C-1001)"}
                },
                "required": ["customer_id"],
            },
        },
    },
    {
        "name": "process-refund",
        "lambda_arn": PROCESS_REFUND_ARN,
        "tool": {
            "name": "process_refund",
            "description": "Process a refund for an order",
            "inputSchema": {
                "type": "object",
                "properties": {
                    "order_id": {"type": "string", "description": "The order ID to refund"},
                    "amount": {"type": "number", "description": "The refund amount in dollars"},
                },
                "required": ["order_id", "amount"],
            },
        },
    },
]

for t in targets:
    target_config = {
        "mcp": {
            "lambda": {
                "lambdaArn": t["lambda_arn"],
                "toolSchema": {"inlinePayload": [t["tool"]]},
            }
        }
    }
    # The Gateway validates that its execution role can invoke the Lambda. The
    # role's inline policy may take a few seconds to propagate after Module's
    # create/put_role_policy, so retry the transient "role lacks permission"
    # ValidationException with a short backoff.
    for attempt in range(6):
        try:
            control_client.create_gateway_target(
                gatewayIdentifier=GATEWAY_ID,
                name=t["name"],
                description=f"{t['tool']['name']} tool",
                targetConfiguration=target_config,
                credentialProviderConfigurations=[{"credentialProviderType": "GATEWAY_IAM_ROLE"}],
            )
            print(f"  ✅ target created: {t['name']}")
            break
        except control_client.exceptions.ConflictException:
            print(f"  • target already exists: {t['name']}")
            break
        except control_client.exceptions.ValidationException as e:
            if "lacks permission" in str(e) and attempt < 5:
                time.sleep(10)  # wait for the role policy to propagate, then retry
                continue
            raise

# Wait for targets to become READY (fail loudly so Module 4.2 doesn't get partial tools).
print("\nWaiting for targets to be READY...")
targets_ready = False
for _ in range(20):
    items = control_client.list_gateway_targets(gatewayIdentifier=GATEWAY_ID)["items"]
    statuses = [it["status"] for it in items]
    if any(s in ("FAILED", "UPDATE_UNSUCCESSFUL") for s in statuses):
        bad = [it["name"] for it in items if it["status"] in ("FAILED", "UPDATE_UNSUCCESSFUL")]
        raise RuntimeError(f"Gateway target(s) failed to register: {bad}")
    if statuses and all(s == "READY" for s in statuses):
        targets_ready = True
        print(f"  All {len(statuses)} targets READY ✅")
        break
    time.sleep(5)
if not targets_ready:
    raise TimeoutError("Gateway targets did not become READY within ~100s — check the console.")


---

## Step 6: Verify the Deployment

Test the Lambda functions directly to confirm they work before connecting via Gateway.

In [ ]:
# Test the Lambda functions directly
lambda_client = boto3.client("lambda", region_name=REGION)

# Test lookup_customer
response = lambda_client.invoke(
    FunctionName="customer-service-lookup-customer",
    Payload=json.dumps({"customer_id": "C-1001"}),
)
result = json.loads(response["Payload"].read())
print("lookup_customer('C-1001'):")
print(f"  {result['result']}")

print()

# Test get_order_history
response = lambda_client.invoke(
    FunctionName="customer-service-get-order-history",
    Payload=json.dumps({"customer_id": "C-1001"}),
)
result = json.loads(response["Payload"].read())
print("get_order_history('C-1001'):")
print(f"  {result['result']}")

print()

# Test process_refund
response = lambda_client.invoke(
    FunctionName="customer-service-process-refund",
    Payload=json.dumps({"order_id": "ORD-5521", "amount": 79.99}),
)
result = json.loads(response["Payload"].read())
print("process_refund('ORD-5521', 79.99):")
print(f"  {result['result']}")

---

## What's Next

Your tools are deployed as Lambda functions and registered with AgentCore Gateway. In **Module 4.2**, you'll connect a Strands agent to these tools using an IAM-authenticated MCP client — no API keys, just IAM roles.